<a href="https://colab.research.google.com/github/tivishs/credit-risk-cutoff-analysis/blob/main/Credit_Risk_Cutoff_Analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Credit Risk Cutoff Analysis

**Data:** Kaggle "Give Me Some Credit", 150,000 borrowers (149,999 after removing one invalid age record). Default means 90+ days late within two years; the overall default rate is 6.7%.

1. **Cleaning:** Monthly income was missing for 29,731 borrowers (about 20%). I flagged those rows first, then filled the blanks with the median ($5,400), because a few extreme incomes (up to $3M a month) distort the average.
2. **Risk drivers:** Default falls with age, from 11.6% (21 to 30) to 2.3% (71+), and rises sharply with credit utilization, from 2.1% to 37.2% for borrowers above 100%. Borrowers ever 90+ days late default at 41.6% against 4.6% for everyone else.
3. **Risk tiers:** With no credit score in the data, I built a three-tier proxy from utilization and late-payment history. It separates risk clearly: Low 1.5% (94,324 borrowers), Medium 10.0% (45,029), High 38.5% (10,646).
4. **Cutoff trade-off:** Approving Low only gives 62.9% approval at 1.5% defaults. Adding Medium lifts approval to 92.9% at 4.3% defaults, but the added group alone defaults at 10.0%.
5. **Recommendation:** Decline High in all cases. Whether to approve Medium depends on its break-even default rate; with illustrative economics it sits just inside break-even, which supports a controlled test on a random sample rather than a full rollout.

## Setup

In [2]:
import pandas as pd

df = pd.read_csv("/content/cs-training.csv", index_col=0)

df.columns = [
    "defaulted",            # 1 = seriously late on a payment within 2 years
    "credit_utilization",   # share of credit limit being used
    "age",
    "late_30_59_days",      # times 30 to 59 days late
    "debt_ratio",           # monthly debt payments / monthly income
    "monthly_income",
    "open_credit_lines",
    "late_90_days",         # times 90+ days late
    "real_estate_loans",
    "late_60_89_days",      # times 60 to 89 days late
    "dependents",
]

df.shape

(150000, 11)

## Part 1: Explore the data

In [3]:
df.head()

,defaulted,credit_utilization,age,late_30_59_days,debt_ratio,monthly_income,open_credit_lines,late_90_days,real_estate_loans,late_60_89_days,dependents
1,1,0.766127,45,2,0.802982,9120.0,13,0,6,0,2.0
2,0,0.957151,40,0,0.121876,2600.0,4,0,0,0,1.0
3,0,0.658180,38,1,0.085113,3042.0,2,1,0,0,0.0
4,0,0.233810,30,0,0.036050,3300.0,5,0,0,0,0.0
5,0,0.907239,49,1,0.024926,63588.0,7,0,1,0,0.0


In [4]:
df["defaulted"].mean()

np.float64(0.06684)

In [5]:
df.isnull().sum()

,0
defaulted,0
credit_utilization,0
age,0
late_30_59_days,0
debt_ratio,0
monthly_income,29731
open_credit_lines,0
late_90_days,0
real_estate_loans,0
late_60_89_days,0


In [6]:
df["monthly_income"].describe().apply(lambda x: f"{x:,.0f}")

,monthly_income
count,"120,269"
mean,"6,670"
std,"14,385"
min,0
25%,"3,400"
50%,"5,400"
75%,"8,249"
max,"3,008,750"


In [7]:
df["age"].describe()

,age
count,150000.000000
mean,52.295207
std,14.771866
min,0.000000
25%,41.000000
50%,52.000000
75%,63.000000
max,109.000000


**First look**
- 150,000 borrowers; 6.7% defaulted within two years.
- Income is missing for 29,731 borrowers (about 1 in 5).
- Data problems: one borrower has age 0, and the top income is $3M a month.
- Debt ratio is unreliable wherever income is missing.

In [8]:
(df["age"] == 0).sum()

np.int64(1)

One borrower has age 0, a data error. Everyone else is 21 or older, so that row is removed.

In [9]:
df = df[df["age"] > 0]
df.shape

(149999, 11)

### Default rate by age

In [10]:
df["age_band"] = pd.cut(df["age"],
                        bins=[20, 30, 40, 50, 60, 70, 110],
                        labels=["21-30", "31-40", "41-50", "51-60", "61-70", "71+"])
df.groupby("age_band", observed=True)["defaulted"].mean()

,defaulted
age_band,
21-30,0.115646
31-40,0.098196
41-50,0.082570
51-60,0.061742
61-70,0.034714
71+,0.022567


Default rate falls steadily with age: highest in the 21-30 band, lowest for 71+.

### Default rate by credit utilization

In [11]:
df["utilization_band"] = pd.cut(df["credit_utilization"],
                                bins=[-0.01, 0.25, 0.50, 0.75, 1.00, 1000000],
                                labels=["0-25%", "26-50%", "51-75%", "76-100%", "Over 100%"])
df.groupby("utilization_band", observed=True)["defaulted"].agg(["count", "mean"])

,count,mean
utilization_band,,
0-25%,87657,0.021367
26-50%,21055,0.052909
51-75%,13764,0.101279
76-100%,24202,0.182134
Over 100%,3321,0.372478


Default rate rises sharply with utilization: 2.1% under 25% used, 18.2% at 76-100%, 37.2% over the limit.
Utilization separates risk much more strongly than age (17x spread vs 5x).

### Default rate by 90+ day late history

In [12]:
df["ever_90_days_late"] = df["late_90_days"] > 0
df.groupby("ever_90_days_late")["defaulted"].agg(["count", "mean"])

,count,mean
ever_90_days_late,,
False,141661,0.046265
True,8338,0.416407


Borrowers who have been 90+ days late default at 41.6% vs 4.6% for everyone else (9x).
They are 5.6% of borrowers but about 35% of all defaults.

## Part 2: Missing income

Flag the missing rows first, so we still know which incomes are estimates after filling.

In [13]:
df["income_missing"] = df["monthly_income"].isnull()
print(df["income_missing"].sum())

29731


Fill with the median rather than the average, because a few extreme incomes pull the average up.

In [14]:
income_median = df["monthly_income"].median()
print(income_median)

5400.0


In [15]:
df["monthly_income"] = df["monthly_income"].fillna(income_median)
print(df["monthly_income"].isnull().sum())
print(df["income_missing"].sum())

0
29731


## Part 3: Risk tiers

There is no credit score in the data, so this builds a simple proxy from the two strongest signals above.

| Tier | Rule |
|---|---|
| High | Ever 90+ days late, or utilization over 100% |
| Medium | Not High, but ever 30 to 89 days late, or utilization over 50% |
| Low | Everyone else |

Everyone starts as Low; Medium is applied next, then High last, so anyone meeting both rules ends up High.

In [16]:
df["risk_tier"] = "Low"
print(df["risk_tier"].value_counts())

risk_tier
Low    149999
Name: count, dtype: int64


In [17]:
df.loc[(df["late_30_59_days"] > 0) | (df["late_60_89_days"] > 0) | (df["credit_utilization"] > 0.5), "risk_tier"] = "Medium"
print(df["risk_tier"].value_counts())

risk_tier
Low       95387
Medium    54612
Name: count, dtype: int64


In [18]:
df.loc[(df["late_90_days"] > 0) | (df["credit_utilization"] > 1), "risk_tier"] = "High"
print(df["risk_tier"].value_counts())

risk_tier
Low       94324
Medium    45029
High      10646
Name: count, dtype: int64


In [19]:
print(df.groupby("risk_tier")["defaulted"].mean())

risk_tier
High      0.385403
Low       0.014906
Medium    0.100313
Name: defaulted, dtype: float64


## Part 4: Approval cutoffs

### Approve Low only

In [20]:
approved = df[df["risk_tier"] == "Low"]
print(len(approved) / len(df))
print(approved["defaulted"].mean())

0.6288308588723924
0.014906068444934481


### Approve Low and Medium

In [21]:
approved = df[df["risk_tier"].isin(["Low", "Medium"])]
print(len(approved) / len(df))
print(approved["defaulted"].mean())

0.9290261935079567
0.042503570070253245
